In [ ]:
import pandas as pd
import numpy as np

# --- 1. 文件路径配置 ---
mapping_file = 'glassdoor_to_wrds_mapping_filtered.csv'
compustat_file = 'compustat_quarterly.csv'
output_file = 'financial_panel_data_compustat_only.csv'

# --- 2. 加载本地数据文件 ---
print("--- 正在加载本地数据文件... ---")
try:
    df_mapping = pd.read_csv(mapping_file)
    df_comp_raw = pd.read_csv(compustat_file)
    print("文件 'glassdoor_to_wrds_mapping_filtered.csv' 和 'compustat_quarterly.csv' 加载成功。")
except FileNotFoundError as e:
    print(f"错误：文件未找到 -> {e}")
    print("请确保这两个CSV文件都在当前目录下。")
    exit()

# --- 3. 规范化和预处理 Compustat 数据 ---
print("\n--- 正在处理 Compustat 数据... ---")
# 根据您之前提供的列名进行重命名
compustat_rename_map = {
    'gvkey': 'gvkey',
    'datadate': 'datadate',
    'atq': 'atq',
    'ceqq': 'ceqq',
    'cheq': 'cheq',
    'cshoq': 'cshoq',
    'ibq': 'ibq',
    'intanq': 'intanq',
    'prccq': 'prccq'
}
try:
    df_comp = df_comp_raw.rename(columns=compustat_rename_map)
except KeyError as e:
    print(f"Compustat 重命名失败，找不到列: {e}。请检查 compustat_rename_map。")
    exit()

# 处理日期和数据类型
df_comp['datadate'] = pd.to_datetime(df_comp['datadate'], errors='coerce')
df_comp.dropna(subset=['datadate'], inplace=True)
df_comp['year_month'] = df_comp['datadate'].dt.to_period('M')
print("Compustat 数据规范化完成。")


# --- 4. 计算 Compustat 控制变量 ---
print("\n--- 正在计算 Compustat 控制变量... ---")
# 排序是向前填充(ffill)和计算同比(shift)的关键
df_comp = df_comp.sort_values(by=['gvkey', 'datadate'])

# Log Assets
df_comp['log_assets'] = np.log(df_comp['atq'].replace(0, np.nan))

# Intangible Assets Ratio
df_comp['intangible_asset_ratio'] = df_comp['intanq'] / df_comp['atq'].replace(0, np.nan)

# Book to Market (BTM)
market_cap = df_comp['cshoq'] * df_comp['prccq']
df_comp['book_to_market'] = df_comp['ceqq'] / market_cap.replace(0, np.nan)

# Earnings Surprise
df_comp['ibq_lag4'] = df_comp.groupby('gvkey')['ibq'].shift(4)
df_comp['earnings_surprise'] = (df_comp['ibq'] - df_comp['ibq_lag4']) / market_cap.replace(0, np.nan)

# Cash Holding
df_comp['cash_holding'] = df_comp['cheq'] / df_comp['atq'].replace(0, np.nan)

# 选择我们计算好的变量
comp_vars = [
    'gvkey', 'year_month', 'log_assets', 'intangible_asset_ratio', 
    'book_to_market', 'earnings_surprise', 'cash_holding'
]
df_comp_processed = df_comp[comp_vars].copy()
print("所有 Compustat 控制变量计算完成。")


# --- 5. 创建最终的面板数据框架并合并 ---
print("\n--- 正在创建完整的公司-月份面板骨架... ---")
# 确定研究的时间窗口，与论文一致
start_date = '2017-11'
end_date = '2019-04'
date_range = pd.period_range(start=start_date, end=end_date, freq='M')
unique_gvkeys = df_mapping['gvkey'].unique()

panel_scaffold = pd.DataFrame(
    [(g, d) for g in unique_gvkeys for d in date_range],
    columns=['gvkey', 'year_month']
)
print(f"已创建覆盖 {len(unique_gvkeys)} 家公司和 {len(date_range)} 个月的面板骨架。")

print("--- 正在合并 Compustat 数据到面板... ---")
# 合并 Compustat 数据
df_comp_processed = df_comp_processed.drop_duplicates(subset=['gvkey', 'year_month'], keep='last')
panel_merged = pd.merge(panel_scaffold, df_comp_processed, on=['gvkey', 'year_month'], how='left')

# 向前填充 (Forward Fill)**
# 季度数据只在季度末出现，用最近一季的数据填充接下来的月份。
panel_merged = panel_merged.sort_values(by=['gvkey', 'year_month'])
comp_control_vars = ['log_assets', 'intangible_asset_ratio', 'book_to_market', 'earnings_surprise', 'cash_holding']
panel_merged[comp_control_vars] = panel_merged.groupby('gvkey')[comp_control_vars].ffill()
print("Compustat 季度数据已向前填充。")


# --- 6. 清理并保存最终结果 ---
final_cols = [
    'gvkey', 'year_month', 'log_assets', 'intangible_asset_ratio', 'book_to_market', 
    'earnings_surprise', 'cash_holding'
]
df_final_financial_panel = panel_merged[final_cols].copy()

# 丢弃所有控制变量都为空的行 
df_final_financial_panel.dropna(subset=final_cols[2:], how='all', inplace=True)

df_final_financial_panel.to_csv(output_file, index=False)

print(f"\n--- 成功！阶段三完成。---")
print(f"仅含 Compustat 变量的财务面板数据已保存到 '{output_file}'")
print("\n最终面板数据预览:")
print(df_final_financial_panel.head())
print("\n数据描述性统计:")
print(df_final_financial_panel.describe())

--- 正在加载本地数据文件... ---
文件 'glassdoor_to_wrds_mapping_filtered.csv' 和 'compustat_quarterly.csv' 加载成功。

--- 正在处理 Compustat 数据... ---
Compustat 数据规范化完成。

--- 正在计算 Compustat 控制变量... ---


d:\APP\Miniconda\envs\MLPython\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


所有 Compustat 控制变量计算完成。

--- 正在创建完整的公司-月份面板骨架... ---
已创建覆盖 3875 家公司和 18 个月的面板骨架。
--- 正在合并 Compustat 数据到面板... ---
Compustat 季度数据已向前填充。

--- 成功！阶段三完成。---
仅含 Compustat 变量的财务面板数据已保存到 'financial_panel_data_compustat_only.csv'

最终面板数据预览:
   gvkey year_month  log_assets  intangible_asset_ratio  book_to_market  \
0   1004    2017-11    7.342326                0.103024        0.627756   
1   1004    2017-12    7.342326                0.103024        0.627756   
2   1004    2018-01    7.342326                0.103024        0.627756   
3   1004    2018-02    7.321321                0.097738        0.620505   
4   1004    2018-03    7.321321                0.097738        0.620505   

   earnings_surprise  cash_holding  
0           0.001316      0.017548  
1           0.001316      0.017548  
2           0.001316      0.017548  
3           0.011458      0.022881  
4           0.011458      0.022881  

数据描述性统计:
               gvkey    log_assets  intangible_asset_ratio  book_to_market  \
count   